### Seq2Seq com Attention vs. ARMAX/MOGWO: split 70/15/15 e k-fold

Treina o Seq2Seq de `01_seq2seq_attention.ipynb` no dataset completo (2.208 amostras) em dois esquemas de `03_full_dataset_sample.py` e compara com o ARMAX/MOGWO publicado (sem reajuste) nas mesmas amostras. Análise em `docs/2_split_artigo_vs_dataset_completo.md`.

In [1]:
from pathlib import Path

import lightning as L
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torchmetrics
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
L.seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_PROCESSED = Path("../../data/processed")
DATA_EXTERNAL = Path("../../data/external")
MODELS_DIR = Path("../../models")
FIGURES_DIR = Path("../../reports/figures/training")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_COLS = ["freq_b1", "freq_b2", "freq_b3", "vazao_entrada"]
OUTPUT_COLS = ["vazao_distribuicao", "nivel_res", "pressao"]
FEATURE_COLS = INPUT_COLS + OUTPUT_COLS
N_TARGETS = len(OUTPUT_COLS)
target_idx = [FEATURE_COLS.index(c) for c in OUTPUT_COLS]

INPUT_WINDOW = 24
OUTPUT_WINDOW = 12

/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/05 15:49:28 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Seed set to 42


### Dados

Série completa reconstruída dos parquets do split 70/15/15; os fold assignments do `TimeSeriesSplit` indexam essa mesma série por `sample_idx`.

In [2]:
splits = {
    name: pd.read_parquet(DATA_PROCESSED / f"mimo_full_sample_{name}.parquet")
    for name in ("train", "val", "test")
}
full_df = pd.concat(splits.values(), ignore_index=True)
full_raw = full_df[FEATURE_COLS].to_numpy()
TRAIN_END = len(splits["train"])
VAL_END = TRAIN_END + len(splits["val"])

folds = pd.read_parquet(DATA_PROCESSED / "mimo_full_kfold_assignments.parquet")
holdout = pd.read_parquet(DATA_PROCESSED / "mimo_full_kfold_test.parquet")
HOLDOUT_START = int(holdout["sample_idx"].iloc[0])

print("série:", full_raw.shape)
print("70/15/15:", TRAIN_END, VAL_END - TRAIN_END, len(full_raw) - VAL_END)
print("folds:", folds["fold"].nunique(), "holdout:", len(holdout))

série: (2208, 7)
70/15/15: 1545 331 332
folds: 5 holdout: 200


In [3]:
class WindowDataset(Dataset):
    """Sliding-window (input, target) pairs over a multivariate series.

    `data` holds every feature (inputs + outputs); the target window
    keeps only `target_idx` columns.
    """

    def __init__(
        self,
        data: np.ndarray,
        target_idx: list[int],
        input_window: int,
        output_window: int,
    ) -> None:
        self.data = data
        self.target_idx = target_idx
        self.input_window = input_window
        self.output_window = output_window
        self.n_samples = len(data) - input_window - output_window + 1

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        start = idx
        mid = start + self.input_window
        end = mid + self.output_window
        x = self.data[start:mid]
        y = self.data[mid:end][:, self.target_idx]
        return torch.FloatTensor(x), torch.FloatTensor(y)

### Modelo Seq2Seq com dot-attention

Encoder LSTM sobre a janela de entrada (todas as features); decoder LSTM autoregressivo com dot-attention sobre os estados do encoder, prevendo as `N_TARGETS` saídas passo a passo. Porte multivariado de `data/external/seq2seq attention.ipynb`.

In [4]:
class Encoder(nn.Module):
    def __init__(
        self, input_size: int, hidden_size: int, num_layers: int = 1
    ) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=False,
        )

    def forward(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        outputs, (hidden, cell) = self.lstm(x)
        return outputs, hidden, cell


class DotAttention(nn.Module):
    def __init__(self, hidden_size: int) -> None:
        super().__init__()
        self.fc = nn.Linear(hidden_size * 2, hidden_size)

    def forward(
        self, hidden: torch.Tensor, encoder_outputs: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        hidden_last_layer = hidden[-1]  # [batch, hidden]
        attn_weights = torch.bmm(
            encoder_outputs.permute(1, 0, 2),
            hidden_last_layer.unsqueeze(2),
        ).squeeze(2)
        attn_weights = F.softmax(attn_weights, dim=1)

        context = torch.bmm(
            attn_weights.unsqueeze(1),
            encoder_outputs.permute(1, 0, 2),
        ).squeeze(1)

        output = torch.cat((context, hidden_last_layer), dim=1)
        output = torch.tanh(self.fc(output))
        return output, attn_weights


class Decoder(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_layers: int = 1,
    ) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=False,
        )
        self.attention = DotAttention(hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(
        self,
        x: torch.Tensor,
        hidden: torch.Tensor,
        cell: torch.Tensor,
        encoder_outputs: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        _, (hidden, cell) = self.lstm(x, (hidden, cell))
        attn_output, attn_weights = self.attention(hidden, encoder_outputs)
        prediction = self.fc(attn_output)
        return prediction, hidden, cell, attn_weights


class Seq2SeqAttention(nn.Module):
    """Encoder-decoder LSTM with dot-attention for MIMO forecasting.

    The decoder feeds back its own prediction (padded with zeros on
    the non-target columns, since only `output_size` of `input_size`
    features are predicted) unless teacher forcing kicks in.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        output_window: int,
        num_layers: int = 1,
    ) -> None:
        super().__init__()
        self.encoder = Encoder(input_size, hidden_size, num_layers)
        self.decoder = Decoder(
            input_size, hidden_size, output_size, num_layers
        )
        self.output_window = output_window
        self.input_size = input_size
        self.output_size = output_size

    def forward(
        self,
        source: torch.Tensor,
        target: torch.Tensor | None = None,
        teacher_forcing_ratio: float = 0.5,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        seq_len, batch_size, _ = source.shape

        outputs = torch.zeros(
            self.output_window,
            batch_size,
            self.output_size,
            device=source.device,
        )
        attention_weights = torch.zeros(
            self.output_window, batch_size, seq_len, device=source.device
        )

        encoder_outputs, hidden, cell = self.encoder(source)
        decoder_input = source[-1].unsqueeze(0)  # [1, batch, input_size]

        for t in range(self.output_window):
            decoder_output, hidden, cell, attn_weights = self.decoder(
                decoder_input, hidden, cell, encoder_outputs
            )
            outputs[t] = decoder_output
            attention_weights[t] = attn_weights

            use_teacher_forcing = (
                target is not None
                and torch.rand(1).item() < teacher_forcing_ratio
            )
            if use_teacher_forcing:
                next_input = source[-1].clone()
                next_input[:, : self.output_size] = target[t]
                decoder_input = next_input.unsqueeze(0)
            else:
                next_input = source[-1].clone()
                next_input[:, : self.output_size] = decoder_output
                decoder_input = next_input.unsqueeze(0)

        return outputs, attention_weights

### LightningModule

Wrapper de treino/validação/teste com MSE loss, `torchmetrics.MeanSquaredError`, `MeanAbsoluteError` e `R2Score` (por saída) em cada etapa, `AdamW` + `ReduceLROnPlateau`.

In [5]:
class Seq2SeqLightning(L.LightningModule):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        output_window: int,
        num_layers: int = 1,
        lr: float = 1e-3,
        teacher_forcing_ratio: float = 0.5,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()
        self.model = Seq2SeqAttention(
            input_size=input_size,
            hidden_size=hidden_size,
            output_size=output_size,
            output_window=output_window,
            num_layers=num_layers,
        )
        self.criterion = nn.MSELoss()
        self.teacher_forcing_ratio = teacher_forcing_ratio

        self.train_mse = torchmetrics.MeanSquaredError()
        self.val_mse = torchmetrics.MeanSquaredError()
        self.test_mse = torchmetrics.MeanSquaredError()
        self.train_mae = torchmetrics.MeanAbsoluteError()
        self.val_mae = torchmetrics.MeanAbsoluteError()
        self.test_mae = torchmetrics.MeanAbsoluteError()
        self.train_r2 = torchmetrics.R2Score(multioutput="raw_values")
        self.val_r2 = torchmetrics.R2Score(multioutput="raw_values")
        self.test_r2 = torchmetrics.R2Score(multioutput="raw_values")

    def _shared_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], teacher_forcing: float
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x, y = batch
        x = x.permute(1, 0, 2)  # [seq_len, batch, features]
        y = y.permute(1, 0, 2)  # [output_window, batch, targets]
        outputs, _ = self.model(x, y, teacher_forcing_ratio=teacher_forcing)
        loss = self.criterion(outputs, y)
        return loss, outputs.contiguous(), y.contiguous()

    def _log_r2(
        self,
        stage: str,
        r2_metric: torchmetrics.R2Score,
        outputs: torch.Tensor,
        y: torch.Tensor,
    ) -> None:
        # R2Score expects [N, num_outputs]; flatten output_window+batch.
        flat_outputs = outputs.reshape(-1, outputs.shape[-1])
        flat_y = y.reshape(-1, y.shape[-1])
        r2_metric(flat_outputs, flat_y)
        for name, value in zip(OUTPUT_COLS, r2_metric.compute()):
            self.log(f"{stage}_r2_{name}", value, on_epoch=True, on_step=False)

    def training_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int
    ) -> torch.Tensor:
        loss, outputs, y = self._shared_step(batch, self.teacher_forcing_ratio)
        self.train_mse(outputs, y)
        self.train_mae(outputs, y)
        self._log_r2("train", self.train_r2, outputs, y)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_mse", self.train_mse, on_epoch=True, on_step=False)
        self.log("train_mae", self.train_mae, on_epoch=True, on_step=False)
        return loss

    def validation_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int
    ) -> torch.Tensor:
        loss, outputs, y = self._shared_step(batch, teacher_forcing=0.0)
        self.val_mse(outputs, y)
        self.val_mae(outputs, y)
        self._log_r2("val", self.val_r2, outputs, y)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mse", self.val_mse, on_epoch=True, on_step=False)
        self.log("val_mae", self.val_mae, on_epoch=True, on_step=False)
        return loss

    def test_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int
    ) -> torch.Tensor:
        loss, outputs, y = self._shared_step(batch, teacher_forcing=0.0)
        self.test_mse(outputs, y)
        self.test_mae(outputs, y)
        self._log_r2("test", self.test_r2, outputs, y)
        self.log("test_loss", loss)
        self.log("test_mse", self.test_mse, on_epoch=True, on_step=False)
        self.log("test_mae", self.test_mae, on_epoch=True, on_step=False)
        return loss

    def configure_optimizers(self) -> dict:
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }

### Treino e avaliação

`train_seq2seq` ajusta o `MinMaxScaler` só no treino e usa a validação cronológica para early stopping. `seq2seq_forecast` encadeia janelas não sobrepostas de 12 passos sobre `[start, stop)`, usando as 24 amostras anteriores como contexto.

In [6]:
HIDDEN_SIZE = 64
NUM_LAYERS = 2


def train_seq2seq(
    train_raw: np.ndarray, val_raw: np.ndarray, run_name: str
) -> tuple[Seq2SeqLightning, MinMaxScaler]:
    """Treina um Seq2Seq e devolve o melhor checkpoint (val_mae)."""
    L.seed_everything(SEED)
    scaler = MinMaxScaler().fit(train_raw)
    loaders = [
        DataLoader(
            WindowDataset(
                scaler.transform(raw), target_idx, INPUT_WINDOW, OUTPUT_WINDOW
            ),
            batch_size=32,
            shuffle=shuffle,
        )
        for raw, shuffle in ((train_raw, True), (val_raw, False))
    ]
    model = Seq2SeqLightning(
        input_size=len(FEATURE_COLS),
        hidden_size=HIDDEN_SIZE,
        output_size=N_TARGETS,
        output_window=OUTPUT_WINDOW,
        num_layers=NUM_LAYERS,
    )
    checkpoint = ModelCheckpoint(monitor="val_mae", mode="min", save_top_k=1)
    trainer = L.Trainer(
        max_epochs=200,
        accelerator="auto",
        callbacks=[EarlyStopping(monitor="val_loss", patience=20), checkpoint],
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(
            {
                "input_window": INPUT_WINDOW,
                "output_window": OUTPUT_WINDOW,
                "hidden_size": HIDDEN_SIZE,
                "num_layers": NUM_LAYERS,
                "seed": SEED,
                "n_train": len(train_raw),
                "n_val": len(val_raw),
            }
        )
        trainer.fit(model, *loaders)
    best = Seq2SeqLightning.load_from_checkpoint(checkpoint.best_model_path)
    return best.eval().to(device), scaler


def seq2seq_forecast(
    model: Seq2SeqLightning, scaler: MinMaxScaler, start: int, stop: int
) -> np.ndarray:
    """Forecast encadeado de 12 passos em [start, stop), escala original."""
    scaled = scaler.transform(full_raw)
    preds = []
    for t0 in range(start, stop - OUTPUT_WINDOW + 1, OUTPUT_WINDOW):
        window = torch.FloatTensor(scaled[t0 - INPUT_WINDOW : t0])
        window = window.unsqueeze(1).to(device)  # [seq, batch=1, features]
        with torch.no_grad():
            out, _ = model.model(window, teacher_forcing_ratio=0.0)
        preds.append(out.squeeze(1).cpu().numpy())
    preds = np.concatenate(preds)
    return (
        preds * scaler.data_range_[target_idx] + scaler.data_min_[target_idx]
    )


mlflow.pytorch.autolog()
mlflow.set_experiment("mimo_sys_seq2seq_attention_splits")

2026/10/05 15:49:29 INFO mlflow.tracking.fluent: Experiment with name 'mimo_sys_seq2seq_attention_splits' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/rwp/code/mimo_sys/notebooks/training/mlruns/3', creation_time=1791226169016, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1791226169016, lifecycle_stage='active', name='mimo_sys_seq2seq_attention_splits', tags={}, trace_location=None, workspace='default'>

### ARMAX/MOGWO publicado (Tabela 3), sem reajuste

Dois modos nas mesmas amostras do Seq2Seq:

- `1 passo`: como no artigo, usa o `y` medido nos lags.
- `12 passos`: free-run de 12 passos a partir do último `y` e resíduo medidos, realimentando a própria predição. Mesmo horizonte do Seq2Seq, mas o ARMAX enxerga `u` futuro e o Seq2Seq não.

In [7]:
tabela3 = pd.read_csv(DATA_EXTERNAL / "tabela3_parametros_armax_papper.csv")
params = tabela3["MOGWO"].to_numpy()
U = full_df[INPUT_COLS].to_numpy().T
Y = full_df[OUTPUT_COLS].to_numpy().T


def armax_base(
    p: np.ndarray, out: int, t: int, y: np.ndarray, u: np.ndarray
) -> float:
    """Parte AR + exógena da predição da saída `out` em t (sem MA)."""
    base = out * 24
    others = [o for o in range(3) if o != out]
    a_self = p[base : base + 3]
    a_o1 = p[base + 3 : base + 6]
    a_o2 = p[base + 6 : base + 9]
    b_u = p[base + 9 : base + 21].reshape(4, 3)
    s = 0.0
    for k in range(3):
        s -= a_self[k] * y[out, t - 1 - k]
        s -= a_o1[k] * y[others[0], t - 1 - k]
        s -= a_o2[k] * y[others[1], t - 1 - k]
        s += b_u[:, k] @ u[:, t - 1 - k]
    return s


def armax_forecast(
    p: np.ndarray,
    u: np.ndarray,
    y: np.ndarray,
    e: np.ndarray,
    t0: int,
    horizon: int,
) -> np.ndarray:
    """Free-run de `horizon` passos a partir de t0 (histórico medido)."""
    y_sim = y.copy()
    e_sim = e.copy()
    e_sim[:, t0:] = 0.0  # resíduo futuro tem esperança zero
    preds = np.zeros((3, horizon))
    for h in range(horizon):
        t = t0 + h
        for out in range(3):
            c_e = p[out * 24 + 21 : out * 24 + 24]
            correction = sum(c_e[k] * e_sim[out, t - 1 - k] for k in range(3))
            preds[out, h] = armax_base(p, out, t, y_sim, u) + correction
        y_sim[:, t] = preds[:, h]
    return preds


# Resíduo e(t) = y(t) - base(t), zero nas 3 primeiras amostras
E = np.zeros_like(Y)
for out in range(3):
    for t in range(3, Y.shape[1]):
        E[out, t] = Y[out, t] - armax_base(params, out, t, Y, U)

# Sanity check: free-run de 1 passo == predição one-step do artigo
t0 = VAL_END
check = armax_forecast(params, U, Y, E, t0, 1)[:, 0]
expected = [
    armax_base(params, o, t0, Y, U)
    + sum(params[o * 24 + 21 + k] * E[o, t0 - 1 - k] for k in range(3))
    for o in range(3)
]
assert np.allclose(check, expected)


def armax_predictions(start: int, stop: int) -> tuple[np.ndarray, np.ndarray]:
    """ARMAX 1 passo e 12 passos nas amostras do forecast encadeado."""
    n = (stop - start) // OUTPUT_WINDOW * OUTPUT_WINDOW
    one_step = np.stack(
        [
            armax_forecast(params, U, Y, E, t, 1)[:, 0]
            for t in range(start, start + n)
        ]
    )
    twelve = np.concatenate(
        [
            armax_forecast(params, U, Y, E, t0, OUTPUT_WINDOW).T
            for t0 in range(start, start + n, OUTPUT_WINDOW)
        ]
    )
    return one_step, twelve

In [8]:
def metrics_table(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    """MSE, MAE, R² e SMAPE por saída."""
    err = y_true - y_pred
    ss_tot = ((y_true - y_true.mean(axis=0)) ** 2).sum(axis=0)
    smape = 100 * np.mean(
        2 * np.abs(err) / (np.abs(y_true) + np.abs(y_pred) + 1e-12), axis=0
    )
    return pd.DataFrame(
        {
            "mse": (err**2).mean(axis=0),
            "mae": np.abs(err).mean(axis=0),
            "r2": 1 - (err**2).sum(axis=0) / ss_tot,
            "smape_%": smape,
        },
        index=OUTPUT_COLS,
    )


def evaluate(
    model: Seq2SeqLightning,
    scaler: MinMaxScaler,
    start: int,
    stop: int,
    scheme: str,
    split: str,
) -> pd.DataFrame:
    """Métricas dos 3 modelos em [start, stop), formato longo."""
    seq2seq = seq2seq_forecast(model, scaler, start, stop)
    armax_1, armax_12 = armax_predictions(start, stop)
    y_true = Y[:, start : start + len(seq2seq)].T
    rows = []
    for name, pred in (
        ("ARMAX 1 passo", armax_1),
        ("ARMAX 12 passos", armax_12),
        ("Seq2Seq 12 passos", seq2seq),
    ):
        table = metrics_table(y_true, pred).rename_axis("saida").reset_index()
        rows.append(table.assign(scheme=scheme, split=split, modelo=name))
    return pd.concat(rows, ignore_index=True)

### Esquema 1: split cronológico 70/15/15

In [9]:
model_70, scaler_70 = train_seq2seq(
    full_raw[:TRAIN_END], full_raw[TRAIN_END:VAL_END], "70_15_15"
)
results = [
    evaluate(model_70, scaler_70, VAL_END, len(full_raw), "70/15/15", "teste")
]

Seed set to 42


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True


INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


2026/10/05 15:49:29 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/mlflow/pytorch/_lightning_autolog.py:542: UserWarning: Autologging is known to be compatible with pytorch-lightning versions between 2.4.0 and 2.6.5 and may not succeed with packages outside this range."


...

/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/rwp/code/mimo_sys/notebooks/training/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


2026/10/05 15:51:06 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


2026/10/05 15:51:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


### Esquema 2: k-fold (`TimeSeriesSplit`, 5 folds) + holdout

Em cada fold, os últimos 15% do treino do fold viram validação de early stopping; a validação do fold e o holdout fixo (200 últimas amostras) são só de avaliação.

In [10]:
for fold, part in folds.groupby("fold"):
    tr = np.sort(part.loc[part["role"] == "train", "sample_idx"].to_numpy())
    va = np.sort(part.loc[part["role"] == "val", "sample_idx"].to_numpy())
    n_es = int(0.15 * len(tr))
    fit_idx, es_idx = tr[:-n_es], tr[-n_es:]
    model, scaler = train_seq2seq(
        full_raw[fit_idx], full_raw[es_idx], f"kfold_{fold}"
    )
    for split, (start, stop) in {
        "val_fold": (int(va[0]), int(va[-1]) + 1),
        "holdout": (HOLDOUT_START, len(full_raw)),
    }.items():
        res = evaluate(model, scaler, start, stop, "kfold", split)
        results.append(res.assign(fold=fold))
    print(f"fold {fold}: treino={len(fit_idx)} es={n_es} val={len(va)}")

Seed set to 42


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True


INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


2026/10/05 15:51:12 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/mlflow/pytorch/_lightning_autolog.py:542: UserWarning: Autologging is known to be compatible with pytorch-lightning versions between 2.4.0 and 2.6.5 and may not succeed with packages outside this range."


...

fold 3: treino=1139 es=201 val=334


/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


2026/10/05 15:54:35 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


2026/10/05 15:54:41 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


fold 4: treino=1423 es=251 val=334


### Resultados

In [11]:
results_df = pd.concat(results, ignore_index=True)
summary = (
    results_df.groupby(["scheme", "split", "saida", "modelo"])[
        ["mse", "mae", "r2", "smape_%"]
    ]
    .agg(["mean", "std"])
    .round(4)
)
summary

mse            \
                                                            mean       std   
scheme   split    saida              modelo                                  
70/15/15 teste    nivel_res          ARMAX 1 passo        0.0034       NaN   
                                     ARMAX 12 passos      0.0353       NaN   
                                     Seq2Seq 12 passos    0.0282       NaN   
                  pressao            ARMAX 1 passo        4.7234       NaN   
                                     ARMAX 12 passos     12.2298       NaN   
                                     Seq2Seq 12 passos    4.9822       NaN   
                  vazao_distribuicao ARMAX 1 passo       53.6091       NaN   
                                     ARMAX 12 passos    270.3170       NaN   
                                     Seq2Seq 12 passos   77.0344       NaN   
kfold    holdout  nivel_res          ARMAX 1 passo        0.0030    0.0000   
                                     ARMAX 12 passos      0.0313    0.0000   
                                     Seq2Seq 12 passos    0.0436    0.0109   
                  pressao            ARMAX 1 passo        6.2947    0.0000   
                                     ARMAX 12 passos     13.7863    0.0000   
                                     Seq2Seq 12 passos    7.4822    3.2003   
                  vazao_distribuicao ARMAX 1 passo       61.9713    0.0000   
                                     ARMAX 12 passos    238.0571    0.0000   
                                     Seq2Seq 12 passos  155.5325  153.7824   
         val_fold nivel_res          ARMAX 1 passo        0.0046    0.0027   
                                     ARMAX 12 passos      0.0785    0.0818   
                                     Seq2Seq 12 passos    0.0533    0.0244   
                  pressao            ARMAX 1 passo        3.0014    1.2005   
                                     ARMAX 12 passos     10.4469    2.7614   
                                     Seq2Seq 12 passos    6.4275    2.5596   
                  vazao_distribuicao ARMAX 1 passo       77.7169   88.2116   
                                     ARMAX 12 passos    268.6878   29.1428   
                                     Seq2Seq 12 passos  171.8975   96.9174   

                                                            mae          \
                                                           mean     std   
scheme   split    saida              modelo                               
70/15/15 teste    nivel_res          ARMAX 1 passo       0.0403     NaN   
                                     ARMAX 12 passos     0.1345     NaN   
                                     Seq2Seq 12 passos   0.1298     NaN   
                  pressao            ARMAX 1 passo       1.1677     NaN   
                                     ARMAX 12 passos     2.7366     NaN   
                                     Seq2Seq 12 passos   1.4528     NaN   
                  vazao_distribuicao ARMAX 1 passo       4.9268     NaN   
                                     ARMAX 12 passos    12.8180     NaN   
                                     Seq2Seq 12 passos   6.5384     NaN   
kfold    holdout  nivel_res          ARMAX 1 passo       0.0385  0.0000   
                                     ARMAX 12 passos     0.1215  0.0000   
                                     Seq2Seq 12 passos   0.1691  0.0233   
                  pressao            ARMAX 1 passo       1.2997  0.0000   
                                     ARMAX 12 passos     2.9021  0.0000   
                                     Seq2Seq 12 passos   1.9122  0.4647   
                  vazao_distribuicao ARMAX 1 passo       4.8347  0.0000   
                                     ARMAX 12 passos    11.6423  0.0000   
                                     Seq2Seq 12 passos   9.1859  4.6444   
         val_fold nivel_res          ARMAX 1 passo       0.0410  0.0068   
                                     ARMAX 12 passos     0.1687  0.056

In [12]:
r2_table = results_df.pivot_table(
    index=["scheme", "split", "saida"],
    columns="modelo",
    values="r2",
    aggfunc="mean",
).round(3)
r2_table

modelo                                ARMAX 1 passo  ARMAX 12 passos  \
scheme   split    saida                                                
70/15/15 teste    nivel_res                   0.932            0.293   
                  pressao                     0.671            0.148   
                  vazao_distribuicao          0.915            0.570   
kfold    holdout  nivel_res                   0.936            0.337   
                  pressao                     0.592            0.107   
                  vazao_distribuicao          0.899            0.611   
         val_fold nivel_res                   0.929           -0.166   
                  pressao                     0.819            0.372   
                  vazao_distribuicao          0.879            0.589   

modelo                                Seq2Seq 12 passos  
scheme   split    saida                                  
70/15/15 teste    nivel_res                       0.435  
                  pressao                         0.653  
                  vazao_distribuicao              0.877  
kfold    holdout  nivel_res                       0.075  
                  pressao                         0.515  
                  vazao_distribuicao              0.746  
         val_fold nivel_res                       0.193  
                  pressao                         0.604  
                  vazao_distribuicao              0.734

In [13]:
labels = ["Vazão de distribuição (l/s)", "Nível (m)", "Pressão (mca)"]
seq2seq_70 = seq2seq_forecast(model_70, scaler_70, VAL_END, len(full_raw))
armax_1_70, armax_12_70 = armax_predictions(VAL_END, len(full_raw))
idx = VAL_END + np.arange(len(seq2seq_70))
series = {
    "Medido": (Y[:, idx].T, "black"),
    "ARMAX 1 passo": (armax_1_70, "firebrick"),
    "ARMAX 12 passos": (armax_12_70, "darkorange"),
    "Seq2Seq 12 passos": (seq2seq_70, "royalblue"),
}
fig = make_subplots(rows=3, cols=1, subplot_titles=labels)
for i in range(3):
    for name, (arr, color) in series.items():
        fig.add_trace(
            go.Scatter(
                x=idx,
                y=arr[:, i],
                name=name,
                line=dict(color=color, width=1),
                legendgroup=name,
                showlegend=(i == 0),
            ),
            row=i + 1,
            col=1,
        )
fig.update_layout(
    title_text="Teste 70/15/15: ARMAX/MOGWO vs Seq2Seq",
    height=800,
    template="plotly_white",
    hovermode="x unified",
)
fig.show()
torch.save(
    model_70.model.state_dict(),
    MODELS_DIR / "seq2seq_attention_mimo_70_15_15.pth",
)